<a href="https://colab.research.google.com/github/kritikarunam30/CareCaller_ps1/blob/main/CareCaller_ps1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import re
import json

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.metrics import f1_score, recall_score, precision_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier

In [ ]:
# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv('/content/drive/MyDrive/hackathon_train.csv')
df.head()

,call_id,outcome,call_duration,attempt_number,direction,attempted_at,scheduled_at,whisper_status,whisper_mismatch_count,organization_id,...,ticket_cat_openai,ticket_cat_openai_notes,ticket_cat_supabase,ticket_cat_supabase_notes,ticket_cat_scheduler_aws,ticket_cat_scheduler_aws_notes,ticket_cat_other,ticket_cat_other_notes,ticket_raised_at,ticket_resolved_at
0,d98c4fc9-0a28-401e-b3e2-cc364c962664,voicemail,25,2,outbound,2026-03-16T19:10:47+00:00,2026-03-16T19:10:47+00:00,skipped,0,org_syn_001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,58c574ab-336b-449d-a0e0-c6669edd5515,incomplete,48,2,outbound,2026-02-28T04:15:42+00:00,2026-02-28T04:15:42+00:00,completed,0,org_syn_002,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,63bcd0bd-04e7-4c77-82a6-163a9cfa2625,incomplete,65,2,outbound,2026-01-19T16:17:40+00:00,2026-01-19T16:17:40+00:00,completed,0,org_syn_003,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2bea5559-8540-4b1f-af83-78edc461af43,escalated,145,1,outbound,2026-02-15T16:12:54+00:00,2026-02-15T16:12:54+00:00,completed,0,org_syn_002,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,383d9034-82d5-404b-998b-8626acc435d1,opted_out,25,1,outbound,2026-01-31T02:59:10+00:00,2026-01-31T02:59:10+00:00,completed,0,org_syn_002,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# =========================
# 2. BASIC CLEANING
# =========================
# Normalize column names just in case
df.columns = [c.strip() for c in df.columns]

# Target column
TARGET = "has_ticket"

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found in dataset.")

# Convert target to numeric if needed
# Handles True/False, yes/no, 1/0 etc.
df[TARGET] = df[TARGET].replace({
    True: 1, False: 0,
    "True": 1, "False": 0,
    "true": 1, "false": 0,
    "yes": 1, "no": 0,
    "Yes": 1, "No": 0
})

df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df[df[TARGET].notna()].copy()
df[TARGET] = df[TARGET].astype(int)


/tmp/ipykernel_8222/3699002837.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[TARGET] = df[TARGET].replace({


In [ ]:
# =========================
# 3. DROP LEAKY COLUMNS
# =========================
leakage_cols = [
    "ticket_has_reason",
    "ticket_priority",
    "ticket_status",
    "ticket_initial_notes",
    "ticket_resolution_notes",
    "ticket_cat_audio_issue",
    "ticket_cat_audio_notes",
    "ticket_cat_elevenlabs",
    "ticket_cat_elevenlabs_notes",
    "ticket_cat_openai",
    "ticket_cat_openai_notes",
    "ticket_cat_supabase",
    "ticket_cat_supabase_notes",
    "ticket_cat_scheduler_aws",
    "ticket_cat_scheduler_aws_notes",
    "ticket_cat_other",
    "ticket_cat_other_notes",
    "ticket_raised_at",
    "ticket_resolved_at",
]

drop_also = [
    "call_id",
    "patient_name_anon",
]

df = df.drop(columns=[c for c in leakage_cols + drop_also if c in df.columns], errors="ignore")



In [ ]:
# =========================
# 4. HELPER FUNCTIONS
# =========================
def safe_to_numeric(dataframe, col):
    if col in dataframe.columns:
        dataframe[col] = pd.to_numeric(dataframe[col], errors="coerce")


def normalize_text(x):
    """Convert any value to clean text."""
    if pd.isna(x):
        return ""
    text = str(x).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def flatten_json_text(x):
    """
    Convert JSON-like string/dict into plain text so TF-IDF can learn from it.
    Works even if responses_json is malformed.
    """
    if pd.isna(x):
        return ""
    if isinstance(x, dict):
        return " ".join([f"{k} {v}" for k, v in x.items()])

    x = str(x).strip()
    if not x:
        return ""

    try:
        obj = json.loads(x)
        if isinstance(obj, dict):
            return " ".join([f"{k} {v}" for k, v in obj.items()])
        elif isinstance(obj, list):
            return " ".join([str(item) for item in obj])
        return str(obj)
    except Exception:
        return x


def preprocess_dataframe(df_input):
    """Apply feature engineering consistently."""
    df = df_input.copy()

    numeric_candidates = [
        "call_duration",
        "attempt_number",
        "whisper_mismatch_count",
        "question_count",
        "answered_count",
        "response_completeness",
        "turn_count",
        "user_turn_count",
        "agent_turn_count",
        "user_word_count",
        "agent_word_count",
        "avg_user_turn_words",
        "avg_agent_turn_words",
        "interruption_count",
        "max_time_in_call",
        "hour_of_day",
        "day_of_week",
        "form_submitted",
    ]

    for col in numeric_candidates:
        safe_to_numeric(df, col)

    # Datetime features
    for dt_col in ["attempted_at", "scheduled_at"]:
        if dt_col in df.columns:
            df[dt_col] = pd.to_datetime(df[dt_col], errors="coerce")
            df[f"{dt_col}_hour"] = df[dt_col].dt.hour
            df[f"{dt_col}_dayofweek"] = df[dt_col].dt.dayofweek
            df[f"{dt_col}_month"] = df[dt_col].dt.month

    # Engineered structured features
    if "question_count" in df.columns and "answered_count" in df.columns:
        df["completion_ratio"] = df["answered_count"] / df["question_count"].replace(0, np.nan)

    if "call_duration" in df.columns and "user_word_count" in df.columns:
        df["user_words_per_sec"] = df["user_word_count"] / df["call_duration"].replace(0, np.nan)

    if "call_duration" in df.columns and "agent_word_count" in df.columns:
        df["agent_words_per_sec"] = df["agent_word_count"] / df["call_duration"].replace(0, np.nan)

    if "user_turn_count" in df.columns and "agent_turn_count" in df.columns:
        df["turn_balance_ratio"] = df["user_turn_count"] / df["agent_turn_count"].replace(0, np.nan)

    if "whisper_mismatch_count" in df.columns:
        df["has_whisper_mismatch"] = (df["whisper_mismatch_count"].fillna(0) > 0).astype(int)

    # =========================
    # RULE-BASED FEATURES
    # =========================
    if "completion_ratio" in df.columns:
        df["rule_low_completion_90"] = (df["completion_ratio"] < 0.90).astype(int)
        df["rule_low_completion_75"] = (df["completion_ratio"] < 0.75).astype(int)
        df["rule_very_low_completion_60"] = (df["completion_ratio"] < 0.60).astype(int)

    if "whisper_mismatch_count" in df.columns:
        df["rule_mismatch_ge_1"] = (df["whisper_mismatch_count"].fillna(0) >= 1).astype(int)
        df["rule_mismatch_ge_2"] = (df["whisper_mismatch_count"].fillna(0) >= 2).astype(int)
        df["rule_mismatch_ge_3"] = (df["whisper_mismatch_count"].fillna(0) >= 3).astype(int)

    if "call_duration" in df.columns:
        df["rule_short_call_30"] = (df["call_duration"] < 30).astype(int)
        df["rule_short_call_60"] = (df["call_duration"] < 60).astype(int)
        df["rule_long_call_600"] = (df["call_duration"] > 600).astype(int)

    if "turn_count" in df.columns:
        df["rule_low_turns_3"] = (df["turn_count"] <= 3).astype(int)
        df["rule_low_turns_5"] = (df["turn_count"] <= 5).astype(int)

    if "interruption_count" in df.columns:
        df["rule_high_interruptions_3"] = (df["interruption_count"].fillna(0) >= 3).astype(int)
        df["rule_high_interruptions_5"] = (df["interruption_count"].fillna(0) >= 5).astype(int)

    if "answered_count" in df.columns:
        df["rule_zero_answers"] = (df["answered_count"].fillna(0) == 0).astype(int)

    if "question_count" in df.columns and "answered_count" in df.columns:
        q = df["question_count"].fillna(0)
        a = df["answered_count"].fillna(0)
        df["rule_many_missed_questions"] = ((q - a) >= 2).astype(int)
        df["rule_half_or_more_missed"] = ((q > 0) & ((q - a) / q >= 0.5)).astype(int)

    # Normalize text columns
    text_cols = ["transcript_text", "whisper_transcript", "validation_notes"]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].apply(normalize_text)

    if "responses_json" in df.columns:
        df["responses_json"] = df["responses_json"].apply(flatten_json_text).apply(normalize_text)

    if "transcript_text" in df.columns:
      txt = df["transcript_text"].fillna("")

      df["rule_kw_not_interested"] = txt.str.contains(r"\bnot interested\b", regex=True).astype(int)
      df["rule_kw_stop_calling"] = txt.str.contains(r"\bstop calling\b|\bdon'?t call\b", regex=True).astype(int)
      df["rule_kw_wrong_number_phrase"] = txt.str.contains(r"\bwrong number\b", regex=True).astype(int)
      df["rule_kw_medical_advice"] = txt.str.contains(
            r"\byou should\b|\bi recommend\b|\btake this medication\b|\byou need to take\b",
            regex=True
        ).astype(int)
      return df


# Apply preprocessing
df = preprocess_dataframe(df)


In [ ]:
# =========================
# 5. SPLIT FEATURES / LABEL
# =========================
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Identify text columns that exist
text_features = [c for c in ["transcript_text", "whisper_transcript", "validation_notes", "responses_json"] if c in X.columns]

# Structured columns = everything else
structured_X = X.drop(columns=text_features, errors="ignore")
numeric_features = structured_X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = structured_X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Text features:", text_features)
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("\nTarget distribution:")
print(y.value_counts())
print(y.value_counts(normalize=True))




Text features: ['transcript_text', 'whisper_transcript', 'validation_notes', 'responses_json']
Numeric features: ['call_duration', 'attempt_number', 'whisper_mismatch_count', 'question_count', 'answered_count', 'response_completeness', 'turn_count', 'user_turn_count', 'agent_turn_count', 'user_word_count', 'agent_word_count', 'avg_user_turn_words', 'avg_agent_turn_words', 'interruption_count', 'max_time_in_call', 'hour_of_day', 'day_of_week', 'attempted_at_hour', 'attempted_at_dayofweek', 'attempted_at_month', 'scheduled_at_hour', 'scheduled_at_dayofweek', 'scheduled_at_month', 'completion_ratio', 'user_words_per_sec', 'agent_words_per_sec', 'turn_balance_ratio', 'has_whisper_mismatch', 'rule_low_completion_90', 'rule_low_completion_75', 'rule_very_low_completion_60', 'rule_mismatch_ge_1', 'rule_mismatch_ge_2', 'rule_mismatch_ge_3', 'rule_short_call_30', 'rule_short_call_60', 'rule_long_call_600', 'rule_low_turns_3', 'rule_low_turns_5', 'rule_high_interruptions_3', 'rule_high_interrupt

In [ ]:

# =========================
# 6. TRAIN / VALID SPLIT
# =========================
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos if pos > 0 else 1.0
print(f"\nscale_pos_weight = {scale_pos_weight:.2f}")


scale_pos_weight = 10.72


In [ ]:
# =========================
# 7. PREPROCESSORS
# =========================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# FunctionTransformer to squeeze single-column dataframe -> 1D array for TF-IDF
def to_1d(x):
    if hasattr(x, "iloc"):
        return x.iloc[:, 0].fillna("").astype(str)
    return pd.Series(x.ravel()).fillna("").astype(str)

text_transformers = []
for col in text_features:
    text_transformers.append(
        (
            f"tfidf_{col}",
            Pipeline(steps=[
                ("selector", FunctionTransformer(to_1d, validate=False)),
                ("tfidf", TfidfVectorizer(
                    max_features=3000,
                    ngram_range=(1, 2),
                    min_df=2,
                    stop_words="english"
                ))
            ]),
            [col]
        )
    )

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        *text_transformers
    ],
    remainder="drop"
)


In [ ]:
# =========================
# 8. MODEL
# =========================
model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    scale_pos_weight=scale_pos_weight
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

In [ ]:
# =========================
# 9. TRAIN
# =========================
pipeline.fit(X_train, y_train)


# =========================
# 10. EVALUATE
# =========================
valid_probs = pipeline.predict_proba(X_valid)[:, 1]

xgb_probs = valid_probs

thresholds = np.arange(0.20, 0.81, 0.05)
#thresholds = np.arange(0.4)
results = []

for t in thresholds:
    preds = (valid_probs >= t).astype(int)
    f1 = f1_score(y_valid, preds, zero_division=0)
    rec = recall_score(y_valid, preds, zero_division=0)
    prec = precision_score(y_valid, preds, zero_division=0)
    results.append((t, f1, rec, prec))




/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['day_of_week']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['day_of_week']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [ ]:
from sklearn.linear_model import LogisticRegression

# Transform features using same preprocessor
X_train_transformed = pipeline.named_steps["preprocessor"].transform(X_train)
X_valid_transformed = pipeline.named_steps["preprocessor"].transform(X_valid)

# Train LR
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_transformed, y_train)

# Get probabilities
lr_probs = lr.predict_proba(X_valid_transformed)[:, 1]

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['day_of_week']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['day_of_week']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# =========================
# ISOLATION FOREST MODEL
# =========================
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    contamination=0.09,
    random_state=42
)

iso.fit(X_train_transformed)

iso_scores = iso.decision_function(X_valid_transformed)

# Normalize (IMPORTANT)
iso_scores = (iso_scores - iso_scores.min()) / (iso_scores.max() - iso_scores.min())

# Convert anomaly → higher probability
iso_probs = 1 - iso_scores

In [ ]:
# =========================
# ENSEMBLE
# =========================
final_probs = (
    0.6 * xgb_probs +   # main model
    0.25 * lr_probs +  # NLP balance
    0.15 * iso_probs   # anomaly boost
)

In [ ]:
best_f1 = 0
best_t = 0.5

for t in thresholds:
    preds = (final_probs >= t).astype(int)
    f1 = f1_score(y_valid, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best Threshold:", best_t)

Best Threshold: 0.39999999999999997


In [ ]:
results_df = pd.DataFrame(results, columns=["threshold", "f1", "recall", "precision"])
print("\nThreshold tuning results:")
print(results_df.sort_values("f1", ascending=False).to_string(index=False))

best_threshold = results_df.sort_values("f1", ascending=False).iloc[0]["threshold"]
best_preds = (valid_probs >= best_threshold).astype(int)

print(f"\nBest threshold: {best_threshold:.2f}")
print("F1 Score  :", f1_score(y_valid, best_preds))
print("Recall    :", recall_score(y_valid, best_preds))
print("Precision :", precision_score(y_valid, best_preds))
print("\nClassification Report:\n", classification_report(y_valid, best_preds))
print("Confusion Matrix:\n", confusion_matrix(y_valid, best_preds))


Threshold tuning results:
 threshold       f1   recall  precision
      0.65 0.956522 0.916667   1.000000
      0.60 0.956522 0.916667   1.000000
      0.55 0.956522 0.916667   1.000000
      0.50 0.956522 0.916667   1.000000
      0.45 0.956522 0.916667   1.000000
      0.75 0.956522 0.916667   1.000000
      0.70 0.956522 0.916667   1.000000
      0.80 0.956522 0.916667   1.000000
      0.40 0.916667 0.916667   0.916667
      0.25 0.880000 0.916667   0.846154
      0.35 0.880000 0.916667   0.846154
      0.30 0.880000 0.916667   0.846154
      0.20 0.846154 0.916667   0.785714

Best threshold: 0.65
F1 Score  : 0.9565217391304348
Recall    : 0.9166666666666666
Precision : 1.0

Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      1.00       126
           1       1.00      0.92      0.96        12

    accuracy                           0.99       138
   macro avg       1.00      0.96      0.98       138
weighted avg    

In [ ]:
# =========================
# LOAD VALIDATION DATA
# =========================
from google.colab import drive
drive.mount('/content/drive')
valid_df = pd.read_csv("/content/drive/MyDrive/hackathon_validation.csv")

# Apply SAME preprocessing function you used for training
valid_df = preprocess_dataframe(valid_df)

# Separate features & target
X_valid_new = valid_df.drop(columns=[TARGET])
y_valid_new = valid_df[TARGET]

# Reindex X_valid_new to match X_train columns
X_valid_aligned = X_valid_new.reindex(columns=X_train.columns, fill_value=0)

# =========================
# PREDICT
# =========================
valid_probs = pipeline.predict_proba(X_valid_aligned)[:, 1]


# =========================
# APPLY THRESHOLD (use best from training)
# =========================
threshold = 0.3   # <-- change this if you found a better one

valid_preds = (valid_probs >= threshold).astype(int)


# =========================
# EVALUATE
# =========================
from sklearn.metrics import f1_score, recall_score, precision_score, classification_report, confusion_matrix

print("F1 Score  :", f1_score(y_valid_new, valid_preds))
print("Recall    :", recall_score(y_valid_new, valid_preds))
print("Precision :", precision_score(y_valid_new, valid_preds))

print("\nClassification Report:\n", classification_report(y_valid_new, valid_preds))
print("Confusion Matrix:\n", confusion_matrix(y_valid_new, valid_preds))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
F1 Score  : 0.8421052631578947
Recall    : 0.7272727272727273
Precision : 1.0

Classification Report:
               precision    recall  f1-score   support

       False       0.98      1.00      0.99       133
        True       1.00      0.73      0.84        11

    accuracy                           0.98       144
   macro avg       0.99      0.86      0.92       144
weighted avg       0.98      0.98      0.98       144

Confusion Matrix:
 [[133   0]
 [  3   8]]


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['day_of_week']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [ ]:
valid_preds_bool = pd.Series(valid_preds).map({1: True, 0: False})

for i, val in valid_preds_bool.items():
    if val:
        print(i, "True")

4 True
28 True
30 True
68 True
95 True
119 True
126 True
142 True


In [ ]:
# =========================
# LOAD TEST DATA
# =========================
test_df = pd.read_csv("/content/drive/MyDrive/hackathon_test.csv")

# Apply SAME preprocessing
test_df = preprocess_dataframe(test_df)

# Keep ID if exists (important for submission)
if "call_id" in test_df.columns:
    test_ids = test_df["call_id"]
else:
    test_ids = pd.Series(range(len(test_df)), name="call_id")


# =========================
# PREDICT
# =========================
X_test = test_df.drop(columns=[TARGET], errors="ignore")

test_probs = pipeline.predict_proba(X_test)[:, 1]

# Use your tuned threshold (adjust if needed)
threshold = 0.4

test_preds = (test_probs >= threshold).astype(int)


# =========================
# CONVERT TO TRUE/FALSE
# =========================
test_preds_bool = pd.Series(test_preds).map({1: True, 0: False})


# =========================
# CREATE SUBMISSION
# =========================
submission = pd.DataFrame({
    "call_id": test_ids,
    "has_ticket": test_preds_bool
})


# =========================
# SAVE FILE
# =========================
submission_path = "/content/drive/MyDrive/submission.csv"
submission.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
print(submission.head())

Submission saved to: /content/drive/MyDrive/submission.csv
                                call_id  has_ticket
0  4a90d9a9-3888-4f8b-8bc0-bb97b3373aab       False
1  2f5d741d-63e0-4764-b8ce-2013bd7ebeea       False
2  4341bf52-4b0a-403c-bcbc-9a8d8ab5198b       False
3  e50cbbce-5741-45cf-993a-a7f7055b79c1       False
4  ba5ed7fc-98ab-470b-89c0-fca5166d3bc2       False


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['day_of_week']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
